# Introduction


## What is Sentiment Analysis?

Sentiment analysis is a Natural Language Processing (NLP) task that identifies the opinion or emotional tone expressed in text. A system may classify a review as **positive**, **negative**, or **neutral**, or use more detailed sentiment categories. In this lab, each app review receives one of two labels: **0 = negative** and **1 = positive**.

The task is not always straightforward. Words can change meaning in context: negation can reverse polarity ("not good"), a review can contain both praise and criticism, and sarcasm may not match the literal words. We will study two ways to handle the task: a lexicon-based method, which uses known word scores and rules, and Naive Bayes, which learns word patterns from reviews with human-assigned labels.


Until now, we've looked at the fundamentals of preprocessing and the use of the bag-of-words approach to model topics in LDA (Latent Dirichlet Allocation).

Now, let's expand on some concepts and revisit, for example, the use of the bag-of-words approach to compare documents. Imagine that we have two documents, and we want to determine whether they discuss the same topic, based on the assumption that if both documents share the same terms, they likely cover the same subject.

With this in mind, we will use the concept of document vectors in the bag-of-words model, which can be explained as representing each document as a vector of word counts or frequencies within a fixed vocabulary. In this vector representation, each dimension corresponds to a specific word in the vocabulary, and the value in each dimension represents how often that word appears in the document.



# Vectors

By transforming documents into these numerical vectors, we can easily compare them using similarity metrics, such as cosine similarity. If two document vectors have high similarity, this suggests that the documents share a significant amount of overlapping content, indicating a related subject. The bag-of-words approach thus provides a straightforward way to quantify the content of documents for comparisons and further analysis in natural language processing tasks.

To compare documents using the Bag of Words approach with CountVectorizer from scikit-learn, we can follow these steps:

1. Transform each document into a word count vector using CountVectorizer.
2. Calculate the similarity between the vectors. One of the most common methods is cosine similarity.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Exemplo de documentos
doc1 = "Eu amo aprender sobre processamento de linguagem natural"
doc2 = "O processamento de linguagem natural é fascinante e cheio de possibilidades"

# Lista de documentos
documents = [doc1, doc2]

# Inicializa o CountVectorizer
vectorizer = CountVectorizer()

# Converte os documentos para a representação bag-of-words
X = vectorizer.fit_transform(documents)

# Calcula a similaridade do cosseno entre os vetores dos documentos
cosine_sim = cosine_similarity(X[0], X[1])

# Exibe a similaridade entre doc1 e doc2
print("Similaridade entre doc1 e doc2:", cosine_sim[0][0])


Here is an example with three documents: two on the same topic (natural language processing) and a third on a different subject (weather). Using the above code calculate the cosine similarity.

In [ ]:
# Exemplo de documentos
doc1 = "O processamento de linguagem natural é fascinante e cheio de possibilidades."
doc2 = "A área de processamento de linguagem natural envolve a análise de textos e fala."
doc3 = "Hoje o clima está ensolarado com algumas nuvens."

from nltk.corpus import stopwords

# Lista de documentos
# documents = [doc1.lower(), doc2.lower(), doc3.lower()]
documents = [doc1, doc2, doc3]

# Inicializa o CountVectorizer
vectorizer = CountVectorizer(lowercase=True,
                             token_pattern=r"\b\w+\b")
                             # stop_words=stopwords.words("portuguese"))

# Converte os documentos para a representação bag-of-words
X = vectorizer.fit_transform(documents)

# print vocabulary
print("Vocabulary:", vectorizer.get_feature_names_out())
print(len(vectorizer.get_feature_names_out()))

# Calcula a similaridade do cosseno entre os documentos
cosine_sim = cosine_similarity(X)

# Exibe a similaridade entre os documentos
print("Similaridade entre doc1 e doc2 (mesmo tema):", cosine_sim[0][1])
print("Similaridade entre doc1 e doc3 (temas diferentes):", cosine_sim[0][2])
print("Similaridade entre doc2 e doc3 (temas diferentes):", cosine_sim[1][2])


Print the obtained vectors and explain the vector in terms of which words are in each position of the vector.

In [ ]:
# TODO: Complete the question above in this code cell.
pass


In [ ]:
# TODO: Complete the question above in this code cell.
pass


In [ ]:
# TODO: Complete the question above in this code cell.
pass


In what range does cosine similarity work?

In [ ]:
# TODO: Complete the question above in this code cell.
pass


In the example above, we used cosine similarity as a metric, but there are other alternatives that can be employed to compare vectors generated by the bag-of-words model. Some of these include Euclidean distance, which measures the straight-line distance between two points in a multi-dimensional space, and Manhattan distance, which calculates the sum of the absolute differences across each dimension. Additionally, the Jaccard similarity coefficient can be used to measure the overlap between two sets, making it particularly useful when focusing on the presence or absence of words rather than their frequency. Each metric has its own advantages depending on the context and desired outcome of the comparison, as some are more sensitive to vector magnitude, while others emphasize the direction or relative frequency of terms.

# Week 6 — Sentiment Analysis and Text Categorization

In this lab, you will classify app reviews as **negative (0)** or **positive (1)** using two approaches:

1. **VADER**, a well-known English sentiment lexicon with rules for cues such as negation, emphasis, and punctuation. It does not learn from this dataset.
2. **Multinomial Naive Bayes**, which learns associations between words and labels from annotated training reviews.

The dataset is loaded once and split once. Both approaches use the **same held-out test reviews and labels**. VADER may leave some reviews neutral, so you will report its coverage as well as its accuracy.


## Shared dataset and train/test split

The CSV contains the review text and its human-assigned polarity. We keep the same rows for both approaches, remove rows missing either field once, and split the data once. We preserve the original review text so VADER can use punctuation and capitalization cues. `CountVectorizer` will apply its own lowercasing when it builds Naive Bayes features.

Run this cell once. The variables `reviews`, `X_train`, `X_test`, `y_train`, and `y_test` are shared below by the lexicon and Naive Bayes sections.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load the CSV exactly once and prepare the shared set of labeled reviews.
data = pd.read_csv("google_play_store_apps_reviews_training.csv")
reviews = data[["review", "polarity"]].dropna().copy()
reviews["review"] = reviews["review"].astype(str).str.strip()

# Create one common held-out test set for both approaches.
X_train, X_test, y_train, y_test = train_test_split(
    reviews["review"],
    reviews["polarity"],
    test_size=0.25,
    random_state=42,
    stratify=reviews["polarity"],
)

print("Shared labeled reviews:", len(reviews))
print("Training reviews:", len(X_train))
print("Shared test reviews:", len(X_test))
print("Labels (0=negative, 1=positive):")
print(reviews["polarity"].value_counts().sort_index())
reviews.head()


## 1 — Lexicon-based sentiment with VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a well-known English lexicon and rule-based sentiment method. It adjusts word valence for some common cues such as negation, intensifiers, capitalization, and punctuation. These rules help with some context, but they do not provide a full understanding of meaning or sarcasm.

NLTK includes `SentimentIntensityAnalyzer`. Its `polarity_scores()` method returns `neg`, `neu`, `pos`, and `compound`. The normalized `compound` score ranges from −1 (most negative) to +1 (most positive). For this binary dataset, use the commonly documented thresholds: compound ≥ 0.05 means positive, compound ≤ −0.05 means negative, and values between these thresholds remain neutral/unclassified. See the [NLTK VADER documentation](https://www.nltk.org/api/nltk.sentiment.vader.html) and [VADER project documentation](https://github.com/cjhutto/vaderSentiment#about-the-scoring).

If NLTK is not installed in your environment, install it before running the next cell with `%pip install nltk`.


### Import and initialize VADER

Complete the imports, download the NLTK VADER lexicon, and create a `SentimentIntensityAnalyzer` object. The analyzer will expose the dictionary through its `lexicon` attribute.


In [ ]:
# TODO: Import nltk and download "vader_lexicon".
# TODO: Import SentimentIntensityAnalyzer from nltk.sentiment.vader.
# TODO: Create an analyzer named vader.

pass


### Inspect some VADER lexicon entries

The lexicon maps individual terms to valence scores. Positive values indicate positive valence and negative values indicate negative valence. Inspect the entries below before applying the analyzer to full reviews.

**Task:** Use `vader.lexicon` to create a small DataFrame with each term and its score. Use `.get()` or check membership so the cell handles a term that is absent from the lexicon.


In [ ]:
sample_terms = ["great", "good", "excellent", "love", "bad", "terrible", "slow", "hate"]

# TODO: Build a DataFrame with columns "term" and "valence" from vader.lexicon.
# Hint: loop over sample_terms and retrieve each term's score.

pass


### Try VADER on short examples

VADER reports several score components; `compound` is the one-dimensional score used for the binary/neutral decision below.

**Task:** For each example, call `vader.polarity_scores(text)` and inspect how the scores change for negation and mixed sentiment.


In [ ]:
examples = [
    "This app is great and helpful.",
    "The update is terrible and slow.",
    "The app is not good.",
    "The app is okay.",
]

# TODO: Loop through examples and print each text with its VADER scores.
pass


## Calculating the Compound Score in VADER

VADER combines adjusted word-valence scores into a **compound score** between −1 and +1. It first aggregates valence, including rule-based adjustments for cues such as negation, intensifiers, capitalization, and punctuation, then normalizes the result.

$$
compound = \frac{S}{\sqrt{S^2 + 15}}
$$

Here, $S$ is the sum of adjusted valence scores. The constant 15 controls how quickly the normalized score approaches the ends of the scale. It is not the number of words or the document length.

**Guided calculation:** for the sentence “The movie is good and fantastic,” use `good = +1.9` and `fantastic = +3.0` as a simplified example. Calculate $S$, then substitute it into the formula. The real analyzer also applies contextual rules, so its output may differ from this simplified calculation.


In [ ]:
import math

# Given simplified valences
valence_good = 1.9
valence_fantastic = 3.0

# TODO: Calculate S as the sum of the two valences.
S = None

# TODO: Apply the normalization formula: S / sqrt(S**2 + 15).
compound_example = None

print("S:", S)
print("Compound score:", compound_example)


### Apply VADER to the shared test reviews

Score `X_test`, the same held-out reviews that Naive Bayes will classify. First inspect the `compound` values. Then assign positive or negative predictions using the thresholds above. Leave reviews in the neutral band unclassified.

**Tasks:**

1. Apply `vader.polarity_scores` to every review in `X_test` and combine the results with `y_test`.
2. Create a nullable integer prediction column. Assign 1 when compound ≥ 0.05 and 0 when compound ≤ −0.05.
3. Calculate VADER coverage and accuracy among covered reviews. Print a confusion matrix with labels `[0, 1]`.
4. Inspect the first few rows and explain one prediction.


In [ ]:
# TODO: Score the shared X_test reviews with vader.polarity_scores.
# TODO: Combine the original text, y_test, and score columns in vader_results.
# TODO: Add a nullable integer column named "vader_prediction".
# TODO: Use the compound thresholds to assign 1 (positive) or 0 (negative).
# TODO: Create vader_covered, calculate coverage and covered-only accuracy.
# TODO: Print the confusion matrix in label order [0, 1].

pass


**Interpretation.** VADER assigns no positive/negative label to reviews in the neutral band. Its accuracy therefore describes only the covered part of the shared test set. Read accuracy together with coverage.

![Precision and Recall](https://upload.wikimedia.org/wikipedia/commons/2/26/Precisionrecall.svg)

## 2 — Text categorization with Multinomial Naive Bayes

Text categorization assigns one of a set of labels to a document. Here, each review is a document and its polarity is the label. `CountVectorizer` turns reviews into a bag-of-words matrix: each column represents a vocabulary term and each cell records its count in a review.

Multinomial Naive Bayes learns how likely words are under each class, then combines these likelihoods with the class prior. Its simplifying assumption is that terms contribute independently given the class. It learns from the training reviews and predicts the same held-out `X_test` used by VADER.


### Inspect the shared split

The shared setup cell already created the train/test split with stratification. Do not split or reload the data again. `X_train` and `y_train` fit Naive Bayes; `X_test` and `y_test` are reserved for evaluation.

**Task:** Print the label counts for train and test. Confirm that `X_test` and `y_test` have the same indices.


In [ ]:
# TODO: Print train and test label counts.
# TODO: Check that X_test.index.equals(y_test.index).
pass


### Convert the shared reviews into count vectors

Fit the vectorizer only on `X_train`. This defines a vocabulary without looking at the test set. Then use `transform()` on `X_test` to apply the same vocabulary to the test reviews.

**Tasks:** Create a `CountVectorizer` using English stop words and `min_df=2`. Fit it on training reviews, transform the test reviews, and inspect the matrix shapes and a few vocabulary terms.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# TODO: Create the vectorizer with stop_words="english" and min_df=2.
# TODO: Fit and transform X_train. Store the result in X_train_vec.
# TODO: Transform X_test using the fitted vectorizer. Store it in X_test_vec.
# TODO: Print the train/test matrix shapes and a few vocabulary terms.

pass


### Train Naive Bayes and evaluate on the shared test set

Fit the classifier with training vectors and labels. Then predict `X_test`, which is identical to the test split scored by VADER.

**Tasks:** Fit `MultinomialNB`, predict the test set, and report accuracy, confusion matrix, and per-class precision/recall/F1. Also calculate Naive Bayes accuracy on `vader_covered` so that both approaches can be compared on the same subset.


In [ ]:
# TODO: Create a MultinomialNB model and fit X_train_vec with y_train.
# TODO: Predict X_test_vec and store predictions as a Series indexed like X_test.
# TODO: Calculate full-test accuracy and print the confusion matrix.
# TODO: Print a classification report for negative (0) and positive (1).
# TODO: If vader_covered is available, calculate NB accuracy on those same reviews.

pass


**Compare the methods.** Both methods use the same test reviews and human labels. Compare accuracy on the subset VADER covers, then consider the coverage difference: Naive Bayes always predicts one of the two classes, while VADER can leave neutral-band cases unclassified. A lexicon is fixed and inspectable; Naive Bayes learns word/class associations from labeled training data.

### Predict new reviews with Naive Bayes

For new text, use the existing vectorizer and model. Do not fit a new vocabulary.

**Task:** Transform the new reviews, predict their polarity, and print each label. Optionally inspect the estimated probabilities. Compare the result with your intuition about the review text.


In [ ]:
new_reviews = [
    "Application is perfect with all the functions you need. I will always recommend it.",
    "I bought it to use as a digital assistant. It doesn't work. And there's no information about it.",
]

# TODO: Transform new_reviews with the fitted vectorizer.
# TODO: Predict labels with the trained model.
# TODO: Print each review with its predicted label.
# Optional: inspect model.predict_proba(new_vectors).

pass


## Final reflection

1. What does VADER provide without training on this dataset?
2. What does Naive Bayes learn from the labeled training reviews?
3. What do coverage and accuracy tell you about VADER on the shared test set?
4. How can negation, sarcasm, domain-specific vocabulary, and class imbalance affect both approaches?
5. Which approach would you choose for a small, labeled app-review project, and why?
